# Урок 02 - Изследване на Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** е унифицирана рамка за изграждане на AI агенти. Тя предоставя чиста, композируема архитектура с четири основни компонента:

- **Клиент** – свързва се с краен пункт на AI модел и управлява комуникацията
- **Агент** – обвива клиент с инструкции и дефиниции на инструменти
- **Инструменти** – разширяват възможностите на агента с персонализирани функции, които моделът може да извиква
- **Сесия** – поддържа история на разговорите за многократни взаимодействия

В този урок ще изградим **агент за резервации на пътувания**, който проверява наличността на дестинации, използвайки тези концепции.


## Настройка


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Разбиране на архитектурата на Agent Framework

Microsoft Agent Framework следва многонишкова архитектура:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клиент** – `FoundryChatClient` се свързва с Azure OpenAI разгръщане. Той се грижи за удостоверяване, форматиране на заявки и анализиране на отговори.
2. **Агент** – Създаден от клиента чрез `provider.create_agent()`, агентът комбинира достъп до модел с инструкции (системен подканващ текст) и инструменти.
3. **Инструменти** – Python функции, декорирани с `@tool`, които агентът може да извиква, за да изпълнява действия или да извлича данни.
4. **Сесия** – Обект `AgentSession` (създаден чрез `agent.create_session()`), който съхранява историята на разговора, позволявайки многократно водене на диалог, където агентът помни предишния контекст.

Нека изградим всеки слой стъпка по стъпка.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Добавяне на инструменти с декоратора @tool

Инструментите позволяват на агентите да предприемат действия, различни от генериране на текст. Декораторът `@tool` превръща обикновена Python функция в нещо, което агентът може да извика.

Основни точки:
- Използвайте `Annotated[type, "description"]`, за да разбере моделът всеки параметър.
- Докстрингът става описание на инструмента, което моделът вижда.
- `approval_mode="never_require"` означава, че инструментът се изпълнява автоматично без потвърждение от потребителя.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Създаване на агент с инструменти

Сега комбинираме клиента, инструкциите и инструментите в агент. `instructions` служат като системна подсказка — те определят личността и поведението на агента.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Многоходови разговори със сесии

Една `AgentSession` (създадена чрез `agent.create_session()`) следи всички съобщения в разговора. Като предаваме същата сесия на всяко извикване на `agent.run()`, агентът има достъп до цялата история на разговора и може да се позове на по-ранни съобщения.

Ние предаваме `tools=[check_destination_availability]`, за да може агентът да извършва проверка на наличността ни при всяка стъпка.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Обобщение

В този урок разгледахте четирите стълба на Microsoft Agent Framework:

| Концепция | Какво научихте |
|---------|------------------|
| **Клиент** | `FoundryChatClient` се свързва с Azure OpenAI с удостоверяване на базата на идентификационни данни |
| **Агент** | `provider.create_agent()` обединява връзка с модел с инструкции и име |
| **Инструменти** | Декораторът `@tool` предоставя Python функции, които агентът може да извиква |
| **Сесия** | `agent.create_session()` поддържа история на разговора през няколко хода |

Тези основни елементи се комбинират, за да създадат агенти, които могат да водят естествени разговори, да извикват външни функции и да поддържат контекст — основата за по-напреднали агентни модели в следващите уроци.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
